# 03 Baseline Model: Popularity-Based Recommender
Evaluation metrics:

- Precision@K
- Recall@K
- NDCG@K

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pickle
import numpy as np
import pandas as pd

from collections import defaultdict

Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pickle
import numpy as np
import pandas as pd

from collections import defaultdict

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load Processed Data

In [3]:
DATASET_DIR = "/content/drive/MyDrive/PixelRec50K"
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")
RESULT_DIR = os.path.join(DATASET_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_5core.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "val_5core.csv"))
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test_5core.csv"))

train_samples = pd.read_csv(os.path.join(PROCESSED_DIR, "train_samples_5core.csv"))
val_samples = pd.read_csv(os.path.join(PROCESSED_DIR, "val_samples_5core.csv"))
test_samples = pd.read_csv(os.path.join(PROCESSED_DIR, "test_samples_5core.csv"))

item_features = pd.read_csv(os.path.join(PROCESSED_DIR, "item_features_5core.csv"))

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Validation samples:", val_samples.shape)
print("Test samples:", test_samples.shape)
display(train_df.head())

Train: (711060, 6)
Validation: (89743, 6)
Test: (113024, 6)
Validation samples: (538458, 3)
Test samples: (678144, 3)


,user_id,item_id,user_idx,item_idx,timestamp,datetime
0,u10002670,i13822,0,7482,1475626862,2016-10-05 00:21:02
1,u10002670,i189468,0,17163,1496056064,2017-05-29 11:07:44
2,u10002670,i215666,0,22298,1513327940,2017-12-15 08:52:20
3,u10002670,i33677,0,32509,1534755859,2018-08-20 09:04:19
4,u10002670,i18543,0,16328,1560431828,2019-06-13 13:17:08


## 2. Train Popularity Baseline

The model learns item popularity from the training set only. Validation and test interactions are not used to compute popularity, which avoids data leakage.

In [4]:
item_popularity = train_df["item_idx"].value_counts()
item_popularity_score = item_popularity / item_popularity.max()

popularity_scores = item_popularity_score.to_dict()

all_items = np.sort(train_df["item_idx"].unique())

print("Number of popular items:", len(popularity_scores))
print("Top 10 most popular item indices:")
display(item_popularity.head(10).to_frame("train_interaction_count"))

Number of popular items: 46180
Top 10 most popular item indices:


,train_interaction_count
item_idx,
12551,132
6251,131
41086,118
20054,117
33444,115
28335,114
10857,113
13935,113
33619,111


## 3. Define Evaluation Metrics

We evaluate the ranking quality of the baseline using Precision@K, Recall@K, and NDCG@K.

In [5]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    if len(recommended_k) == 0:
        return 0.0
    return len(set(recommended_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    return len(set(recommended_k) & set(relevant)) / len(set(relevant))

def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    dcg = 0.0

    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))

    if idcg == 0:
        return 0.0

    return dcg / idcg

## 4. Candidate-Based Evaluation

For efficiency, we evaluate each user on candidate items from the sampled validation/test files. Each user's candidate set contains observed positive items and sampled negative items.

In [6]:
def evaluate_popularity_on_samples(samples_df, k_values=[5, 10, 20]):
    results = []

    for user_idx, user_data in samples_df.groupby("user_idx"):
        candidate_items = user_data["item_idx"].unique().tolist()
        relevant_items = user_data.loc[user_data["label"] == 1, "item_idx"].unique().tolist()

        ranked_items = sorted(
            candidate_items,
            key=lambda item: popularity_scores.get(item, 0.0),
            reverse=True
        )

        for k in k_values:
            results.append({
                "user_idx": user_idx,
                "k": k,
                "precision": precision_at_k(ranked_items, relevant_items, k),
                "recall": recall_at_k(ranked_items, relevant_items, k),
                "ndcg": ndcg_at_k(ranked_items, relevant_items, k)
            })

    results_df = pd.DataFrame(results)

    summary = (
        results_df
        .groupby("k")[["precision", "recall", "ndcg"]]
        .mean()
        .reset_index()
    )

    return summary, results_df

In [7]:
val_summary, val_user_results = evaluate_popularity_on_samples(val_samples, k_values=[5, 10, 20])
test_summary, test_user_results = evaluate_popularity_on_samples(test_samples, k_values=[5, 10, 20])

print("Validation Results")
display(val_summary)

print("Test Results")
display(test_summary)

Validation Results


,k,precision,recall,ndcg
0,5,0.191597,0.688617,0.481878
1,10,0.134181,0.887377,0.559032
2,20,0.081423,0.975002,0.591264


Test Results


,k,precision,recall,ndcg
0,5,0.154959,0.448973,0.318011
1,10,0.131894,0.723811,0.425213
2,20,0.095941,0.945747,0.503074


The popularity baseline performs surprisingly well, especially in Recall@10 and Recall@20, indicating strong popularity bias in the dataset. However, because it recommends the same popular items to most users, it does not capture individual preferences. This creates a meaningful benchmark: later models must outperform popularity-based recommendation or provide better personalization and long-tail coverage.

## 5. Generate Example Recommendations

For qualitative inspection, we generate Top-N recommendations for selected users. Items already seen in training are excluded.

In [8]:
global_popular_items = item_popularity.index.tolist()

train_seen_by_user = (
    train_df
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

def recommend_popular_for_user(user_idx, n=10):
    seen_items = train_seen_by_user.get(user_idx, set())
    recommendations = []

    for item in global_popular_items:
        if item not in seen_items:
            recommendations.append(item)

        if len(recommendations) >= n:
            break

    return recommendations

sample_users = test_df["user_idx"].drop_duplicates().sample(5, random_state=42).tolist()

example_recs = []

for user_idx in sample_users:
    recs = recommend_popular_for_user(user_idx, n=10)

    for rank, item_idx in enumerate(recs, start=1):
        example_recs.append({
            "user_idx": user_idx,
            "rank": rank,
            "recommended_item_idx": item_idx,
            "popularity_score": popularity_scores.get(item_idx, 0.0)
        })

example_recs_df = pd.DataFrame(example_recs)
display(example_recs_df.head(20))

,user_idx,rank,recommended_item_idx,popularity_score
0,8194,1,12551,1.000000
1,8194,2,6251,0.992424
2,8194,3,41086,0.893939
3,8194,4,20054,0.886364
4,8194,5,33444,0.871212
5,8194,6,28335,0.863636
6,8194,7,10857,0.856061
7,8194,8,13935,0.856061
8,8194,9,33619,0.840909
9,8194,10,35724,0.840909


## 6. Add Item Metadata to Example Recommendations

We join recommendation examples with item metadata so the results are easier to interpret.

In [9]:
item_info_path = os.path.join(DATASET_DIR, "pixel50k_item_info.csv")
item_info = pd.read_csv(item_info_path)

with open(os.path.join(PROCESSED_DIR, "item_encoder.pkl"), "rb") as f:
    item_encoder = pickle.load(f)

item_mapping = pd.DataFrame({
    "item_id": item_encoder.classes_,
    "item_idx": np.arange(len(item_encoder.classes_))
})

example_recs_with_meta = (
    example_recs_df
    .merge(item_mapping, left_on="recommended_item_idx", right_on="item_idx", how="left")
    .merge(item_info, on="item_id", how="left")
)

cols_to_show = [
    "user_idx",
    "rank",
    "recommended_item_idx",
    "item_id",
    "popularity_score",
    "title",
    "tag"
]

display(example_recs_with_meta[cols_to_show].head(20))

,user_idx,rank,recommended_item_idx,item_id,popularity_score,title,tag
0,8194,1,12551,i164904,1.000000,Russia banned from replacing national anthem w...,Global
1,8194,2,6251,i131669,0.992424,"Woman: ""He's a Moor you have no right to take ...",Daily Life
2,8194,3,41086,i67900,0.893939,This woman is playing poker and her house is g...,Daily Life
3,8194,4,20054,i20446,0.886364,[Fantastic] Uncovered Japanese Internet cafe r...,"Social Sciences, Law, and Psychology"
4,8194,5,33444,i35271,0.871212,"Tencent's ""King's Honor"" is in court: the game...",Society
5,8194,6,28335,i27361,0.863636,"After the tragedy in Chengdu, I see the shadow...",Global
6,8194,7,10857,i15609,0.856061,Van Darkholme returns in DDF form!,Daily Life
7,8194,8,13935,i1721,0.856061,"Chengdu, a medical student to see blood must b...",Hot Topics
8,8194,9,33619,i35565,0.840909,"The rural boy's one-to-one remake of ""Killing ...",Short Film
9,8194,10,35724,i40259,0.840909,Inner Mongolia cultivates tread-resistant lawn...,Society


## 7. Save Baseline Results

In [10]:
val_summary["split"] = "validation"
test_summary["split"] = "test"

baseline_summary = pd.concat([val_summary, test_summary], ignore_index=True)
baseline_summary["model"] = "Popularity Baseline"
baseline_summary["features"] = "Training item interaction count"

baseline_summary = baseline_summary[
    ["model", "features", "split", "k", "precision", "recall", "ndcg"]
]

baseline_summary.to_csv(
    os.path.join(RESULT_DIR, "popularity_baseline_results.csv"),
    index=False
)

example_recs_with_meta.to_csv(
    os.path.join(RESULT_DIR, "popularity_baseline_example_recommendations.csv"),
    index=False
)

display(baseline_summary)

print("Saved results to:", RESULT_DIR)

,model,features,split,k,precision,recall,ndcg
0,Popularity Baseline,Training item interaction count,validation,5,0.191597,0.688617,0.481878
1,Popularity Baseline,Training item interaction count,validation,10,0.134181,0.887377,0.559032
2,Popularity Baseline,Training item interaction count,validation,20,0.081423,0.975002,0.591264
3,Popularity Baseline,Training item interaction count,test,5,0.154959,0.448973,0.318011
4,Popularity Baseline,Training item interaction count,test,10,0.131894,0.723811,0.425213
5,Popularity Baseline,Training item interaction count,test,20,0.095941,0.945747,0.503074


Saved results to: /content/drive/MyDrive/PixelRec50K/results


Although the precision values are modest, the popularity baseline achieves strong recall, especially at larger K. On the test set, Recall@10 reaches 0.724 and Recall@20 reaches 0.946, indicating that many future user interactions are concentrated among globally popular items. This is consistent with the long-tail popularity pattern observed in the EDA.

However, this baseline is non-personalized and recommends similar popular items to most users. Therefore, its high recall does not necessarily mean it provides high-quality personalized recommendations. More advanced models should aim to improve ranking quality, personalization, and long-tail recommendation beyond this popularity-based benchmark.